# Laboratorio — ML para series de tiempo (15 min)
**Curso:** Aprendizaje Automático · MCDA 2026-1
**Sesión:** Series de tiempo con modelos de Machine Learning

## Objetivo
Tomar una serie de tiempo, **construir features supervisadas** (lags, rolling, calendario) y entrenar un modelo de ML para pronosticar. Validar correctamente con un esquema *walk-forward*.

## Dataset
Serie diaria sintética de **demanda de un producto** durante ~3 años, con tendencia, estacionalidad semanal y anual, y ruido. La generamos en la primera celda para que sea reproducible.

## Agenda (15 min)
| Paso | Tiempo | Qué haces |
|------|--------|-----------|
| 1. Generar y graficar la serie | 2 min | Ver tendencia y estacionalidad |
| 2. Construir features (lags, rolling, calendario) | 4 min | Convertir a tabla supervisada |
| 3. Entrenar Random Forest con split temporal | 3 min | `shuffle=False` |
| 4. Métricas y validación walk-forward | 3 min | MAE, MAPE, WMAPE, `TimeSeriesSplit` |
| 5. Retos | 3 min | 3 micro-retos individuales |

## Paso 1 · Generar y graficar la serie (2 min)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
fechas = pd.date_range('2022-01-01', '2024-12-31', freq='D')
t = np.arange(len(fechas))

tendencia    = 0.04 * t
estac_anual  = 12 * np.sin(2 * np.pi * t / 365.25)
estac_semana = 4  * np.sin(2 * np.pi * t / 7) + 2 * np.cos(2 * np.pi * t / 7)
ruido        = rng.normal(0, 2.5, size=len(t))
y = 60 + tendencia + estac_anual + estac_semana + ruido

serie = pd.DataFrame({'fecha': fechas, 'y': y}).set_index('fecha')

print(serie.head())
serie['y'].plot(figsize=(11, 3.2), color='#1FA8DC')
plt.title('Demanda diaria — serie de ejemplo'); plt.ylabel('Unidades'); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

**Pregunta rápida:** ¿qué dos componentes ves a simple vista? ¿Por qué *no* podríamos hacer un `train_test_split` aleatorio?

## Paso 2 · Construir features (4 min)
Convertimos la serie en una tabla `(features, target)` apta para cualquier modelo de ML.

- **Lags**: valores pasados (`y_{t-1}`, `y_{t-7}`, `y_{t-14}`, `y_{t-28}`).
- **Rolling**: media y desviación móvil de los últimos 7 y 28 días.
- **Calendario codificado cíclicamente**: día de semana y día del año vía seno/coseno.
- **Tendencia**: índice numérico del tiempo.

In [ ]:
def construir_features(df, lags=(1, 7, 14, 28), ventanas=(7, 28)):
    out = df.copy()
    # lags
    for L in lags:
        out[f'lag_{L}'] = out['y'].shift(L)
    # rolling sobre el lag 1 (evita filtrar el valor presente)
    for w in ventanas:
        out[f'roll_mean_{w}'] = out['y'].shift(1).rolling(w).mean()
        out[f'roll_std_{w}']  = out['y'].shift(1).rolling(w).std()
    # calendario cíclico
    dow = out.index.dayofweek
    doy = out.index.dayofyear
    out['dow_sin'] = np.sin(2 * np.pi * dow / 7)
    out['dow_cos'] = np.cos(2 * np.pi * dow / 7)
    out['doy_sin'] = np.sin(2 * np.pi * doy / 365.25)
    out['doy_cos'] = np.cos(2 * np.pi * doy / 365.25)
    # tendencia
    out['t'] = np.arange(len(out))
    return out.dropna()

tabla = construir_features(serie)
print('Forma:', tabla.shape)
tabla.head()

## Paso 3 · Entrenar Random Forest con split temporal (3 min)
Cortamos por **fecha**, no aleatoriamente. El test deben ser los últimos 90 días.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

X = tabla.drop(columns='y')
y_full = tabla['y']

# Split temporal: últimos 90 días = test
corte = tabla.index.max() - pd.Timedelta(days=90)
X_train, X_test = X.loc[:corte], X.loc[corte + pd.Timedelta(days=1):]
y_train, y_test = y_full.loc[:corte], y_full.loc[corte + pd.Timedelta(days=1):]
print(f'Train: {len(X_train)} días  ({X_train.index.min().date()} → {X_train.index.max().date()})')
print(f'Test : {len(X_test)} días  ({X_test.index.min().date()} → {X_test.index.max().date()})')

rf = RandomForestRegressor(n_estimators=400, min_samples_leaf=3, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

# Visualización
plt.figure(figsize=(11, 3.5))
plt.plot(y_train.index[-180:], y_train.iloc[-180:], color='#888', label='Train (últimos 180d)')
plt.plot(y_test.index, y_test,  color='#0F1B5E', label='Real (test)')
plt.plot(y_test.index, y_pred,  color='#F0C300', label='Pronóstico RF')
plt.title('Pronóstico vs. realidad (90 días)'); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Paso 4 · Métricas y validación walk-forward (3 min)
Usamos métricas típicas de pronóstico y validamos sobre **varias ventanas** con `TimeSeriesSplit`.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit

def mape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def wmape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print('--- Test (últimos 90 días) ---')
print(f'MAE  : {mean_absolute_error(y_test, y_pred):.2f}')
print(f'RMSE : {rmse:.2f}')
print(f'MAPE : {mape(y_test, y_pred):.2f} %')
print(f'WMAPE: {wmape(y_test, y_pred):.2f} %')

In [ ]:
# Walk-forward: 5 cortes, cada uno entrena hasta cierto día y evalúa los siguientes
tscv = TimeSeriesSplit(n_splits=5, test_size=60)
mapes, wmapes = [], []

for i, (tr, te) in enumerate(tscv.split(X), 1):
    m = RandomForestRegressor(n_estimators=300, min_samples_leaf=3,
                              random_state=42, n_jobs=-1)
    m.fit(X.iloc[tr], y_full.iloc[tr])
    pred = m.predict(X.iloc[te])
    mapes.append(mape(y_full.iloc[te], pred))
    wmapes.append(wmape(y_full.iloc[te], pred))
    print(f'Fold {i}: train={len(tr)}d · test={len(te)}d · MAPE={mapes[-1]:.2f}% · WMAPE={wmapes[-1]:.2f}%')

print(f'\nMAPE  promedio: {np.mean(mapes):.2f} %  ± {np.std(mapes):.2f}')
print(f'WMAPE promedio: {np.mean(wmapes):.2f} %  ± {np.std(wmapes):.2f}')

## Paso 5 · Retos individuales (3 min)

### Reto 1 · Importancia de variables
Pinta `pd.Series(rf.feature_importances_, index=X.columns).sort_values().plot.barh()` y di cuáles son los **3 features más importantes**. ¿Domina algún lag en particular?

### Reto 2 · Pronóstico ingenuo (baseline)
Implementa un baseline *naïve estacional*: `y_pred_naive = y_test.shift(7)` (predice con el valor de hace una semana). Calcula su WMAPE. ¿Cuánto le saca el Random Forest al baseline?

### Reto 3 · Cambiar de modelo
Reemplaza el `RandomForestRegressor` por `GradientBoostingRegressor(n_estimators=400, max_depth=3, learning_rate=0.05, random_state=42)`. Reporta MAE y WMAPE en el test. ¿Cuál de los dos modelos eligirías y por qué?

In [ ]:
# Reto 1 — tu código aquí


In [ ]:
# Reto 2 — tu código aquí
# pista: el primer valor del shift cae en NaN; recorta antes de calcular WMAPE


In [ ]:
# Reto 3 — tu código aquí
# from sklearn.ensemble import GradientBoostingRegressor


## Cierre
Convertir una serie en una tabla `(features, target)` con **lags + rolling + calendario cíclico** habilita usar cualquier modelo de ML para pronóstico. Lo crítico no es el modelo: es **respetar el orden temporal** al partir los datos y al validar (walk-forward), y elegir métricas que tengan sentido para el negocio (WMAPE suele ser más útil que MAPE cuando hay valores cercanos a cero).